### Data Reading Json (singleline/multiline json)

In [0]:
df_json=spark.read.format('json').option('inferschema',True)\
.option('header',True)\
.option('multiline',False)\
.load("/Volumes/workspace/default/file/drivers.json")

In [0]:
df_json.display()

##Data Reading

In [0]:
dbutils.fs.ls("dbfs://")

In [0]:

df = spark.read.format('csv').option('inferSchema',True).option('header',True).load("dbfs:/Volumes/workspace/default/file/BigMart Sales (1).csv")

In [0]:
df.display()

### Schema Defination


In [0]:
df.printSchema()

### DDL Schema

In [0]:
my_ddl_Schema='''
                Item_identifier String,
                Item_Weight String,
                Item_Fat_Content String,
                Item_Visibility Double,
                Item_Type String,
                Item_MRP Double,
                Outlet_Identifier String,
                Outlet_Establishment_Year Integer,
                Outlet_Size String,
                Outlet_Location_Type String,
                Outlet_Type String,
                Item_Outlet_Sales Double
             '''

In [0]:
df=spark.read.format('csv').schema(my_ddl_Schema).option('header',True).load("dbfs:/Volumes/workspace/default/file/BigMart Sales (1).csv")


In [0]:
df.display()

In [0]:
df.printSchema()


### StructTye() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
my_struct_schema=StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_Weight',StringType(),True),
     StructField('Item_Fat_Content',StringType(),True),
     StructField('Item_Visibility',DoubleType(),True),
     StructField('Item_Type',StringType(),True),
     StructField('Item_MRP',DoubleType(),True),
     StructField('Outlet_Identifier',StringType(),True),
     StructField('Outlet_Establishment_Year',IntegerType(),True),
     StructField('Outlet_Size',StringType(),True),
     StructField('Outlet_Location_Type',StringType(),True),
     StructField('Outlet_Type',StringType(),True),
     StructField('Item_Outlet_Sales',DoubleType(),True)
])


In [0]:
df=spark.read.format('csv')\
    .schema(my_struct_schema)\
    .option('header',True)\
    .load('dbfs:/Volumes/workspace/default/file/BigMart Sales (1).csv')



In [0]:
df.printSchema()

## Select


In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

In [0]:
df_sel.display()

### COL
to select column while using function like aggregation and standardize way


In [0]:
df.select(col('Item_Identifier')).display()

### ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_Id')).display()

#Filter


###Scenario-1


In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

###Scenario-2


In [0]:
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

###Scenario-3

In [0]:
df.filter((col('Outlet_size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

### withColumnRenamed
this rename col name at dataframe level

In [0]:
df.withColumnRenamed('Item_Weight','Item_wt').display()

#withColumn


####Scenario-1 
Create a new Column with constant value

In [0]:
df=df.withColumn('flag',lit("new"))

In [0]:
df.display()

####Scenario-2 
Create a new column with transform value

In [0]:
df.withColumn('multiply_Wt_Mrp',col('Item_Weight')*col('Item_MRP')).display()

#### Scenario 3: 
Modify the existing column value


In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf")).display()

###Type Casting

In [0]:

df=df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort/OrderBy

###Scenario --1

In [0]:
df.sort(col('Item_Weight').desc()).display()

###Scenario--2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

###Scenario --3

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[0,0]).display()

###Scenario--4

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[0,1]).display()

###Limit


In [0]:
df.limit(2).display()

###Drop

###Scenario-1
to drop column

In [0]:
df.drop('Item_Visibility').display()

###Scenario-2
drop multiple column


In [0]:
df.drop('Item_Visibility','Item_Weight').display()

###Drop duplicates

##Scenario-1
within a tble

In [0]:
##df.dropDuplicates().display()

##Scenario-2
Within specific Column


In [0]:
df.dropDuplicates(subset=["Item_Type"]).display()


##Distinct 
Same as drop duplicate for table


In [0]:
df.distinct().display()

##Union and Union By Name


###Preparing Dataframe

In [0]:
data1=[('1','kadam'),('2','sid')]
Schema1= 'id String,name String'
df1=spark.createDataFrame(data1,Schema1)

data2=[('3','rahul'),('4','jas')]
Schema2='id String,name String'
df2=spark.createDataFrame(data2,Schema2)

In [0]:
df1.display()

In [0]:
df2.display()

####Union

In [0]:
df1.union(df2).display()

###Union by name



In [0]:
data1=[('kadam','1'),('sid','2')]
Schema1= 'name String,id String'
df1=spark.createDataFrame(data1,Schema1)

df1.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

#String Functions

###InitCap() ,lower(),Upper()

In [0]:
df.select(initcap('Item_Type')).display()

In [0]:
df.select(lower('Item_Type')).display()

In [0]:
df.select(upper('Item_Type')).display()

#Date Functions

###Current date


In [0]:
df=df.withColumn('curr_date', current_date())

In [0]:
df.display()

### Date add


In [0]:
df=df.withColumn('week_after', date_add('curr_date', 7))
df.display()

#date_sub()

In [0]:
df=df.withColumn('week_before', date_sub('curr_date', 7))
df.display()

In [0]:
df.withColumn('week_after', date_add('curr_date', -7)).display()

#Datediff()


In [0]:
df= df.withColumn('datediff', datediff('curr_date','week_after'))
df.display()

###Date_format()


In [0]:

df=df.withColumn('week_after', date_format('week_after', 'dd-MM-yyyy'))
df.display()

In [0]:
df.printSchema()

#Handling Null

####dropping nulls

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

####filling null


In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

###Split() and Indexing
Split convert into list and using index we can fetch split values


######split

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

#####index

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

##Explode
change list values to new row


In [0]:
df_exp=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

###Array_Contains
to compare the value in list or 1d

In [0]:
df_exp.withColumn('Type_flag', array_contains('Outlet_type', 'Type1')).display()

###groupby

Scenario-1

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

Scenario 2

In [0]:
df.groupBy("Item_Type").agg(avg("Item_MRP")).display()

scenario-3

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias('Total_Mrp')).display()

Scenario-4

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias('Total_Mrp'),(avg('Item_MRP'))).display()

####CollectList
its like group_concat of sql.This aggregate all values inside the list

In [0]:
data = [
    ('user1', 'book1'),
    ('user1', 'book2'),
    ('user2', 'book2'),
    ('user2', 'book4'),
    ('user3', 'book1')
]

schema = 'user string, book string'

df_book = spark.createDataFrame(data, schema)

df_book.display()

In [0]:
df_book.groupBy("user").agg(collect_list("book")).display()

###Pivot

In [0]:
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

###When-Otherwise
Its same as case when in SQL

#######Scenario_1


In [0]:
df=df.withColumn('Veg_flag',when(col("Item_Type")=='Meat','NonVeg').otherwise('Veg'))
df.display()

#######Scenario-2

In [0]:
df.withColumn(
    'veg_exp_flag',
    when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP') < 100),
        'Veg_Inexpensive'
    )
    .when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP') > 100),
        'Veg_Expensive'
    )
    .otherwise('Non_Veg')
).display()

#joins


In [0]:
data1 = [
    ('1', 'gaur', 'd01'),
    ('2', 'kit', 'd02'),
    ('3', 'sam', 'd03'),
    ('4', 'tim', 'd03'),
    ('5', 'aman', 'd05'),
    ('6', 'jim', 'd06')
]

schema1 = 'emp_id STRING, emp_name STRING, dept_id STRING'

df1 = spark.createDataFrame(data1, schema1)


data2 = [
    ('d01', 'HR'),
    ('d02', 'Marketing'),
    ('d03', 'Accounts'),
    ('d04', 'IT'),
    ('d05', 'Finance')
]

schema2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(data2, schema2)

In [0]:
df1.display()
df2.display()



###InnerJoins


In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'inner').display()

###Left  Join


In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'left').display()

###Right Join

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'right').display()

###Anti Join

if we do join with df1 to df2 and need data which are not present in df2

In [0]:
df1.join(df2, df1.dept_id == df2.dept_id, 'anti').display()

#Windows Function


###RowNumber

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

###Rank

In [0]:
df.withColumn('rankCol', rank().over(Window.orderBy('Item_Identifier'))).display()

###Dense_rank

In [0]:
df.withColumn('denserankCol', dense_rank().over(Window.orderBy('Item_Identifier'))).display()
df.withColumn('denserankCol', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

###Cumulative sum

It sum the data row by row

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_type'))).display()

Cumsum--need to use rowsBetween

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

This  unboundedFollowing is used to get full sum of allm data

In [0]:
df.withColumn('totalSum',sum('Item_MRP').over(Window.orderBy('Item_type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

###UDF ;User defined functions:
We should use less udf cause of perfermance issue.

Step-1
Create python function


In [0]:
def my_func(x):
     return x*x

Step-2

convert python function in pyspark udf


In [0]:
my_udf=udf(my_func)

Step- 3
now pyspark function we can use

In [0]:
df.withColumn('myfunn',my_udf('Item_MRP')).display()

#Data Writing

#####Csv

In [0]:
df.write.format('csv')\
    .save('dbfs:/Volumes/workspace/default/file/data.csv')

####Writing modes


#####Append


In [0]:
df.write.format('csv')\
    .mode('append')\
    .save('dbfs:/Volumes/workspace/default/file/data.csv')

In [0]:
df.write.format('csv')\
    .mode('append')\
    .option('path','dbfs:/Volumes/workspace/default/file/data.csv')\
        .save()

#####OverWrite

In [0]:
df.write.format('csv')\
    .mode('Overwrite')\
    .option('path','dbfs:/Volumes/workspace/default/file/data.csv')\
        .save()

#####Error

In [0]:
df.write.format('csv')\
    .mode('Error')\
    .option('path','dbfs:/Volumes/workspace/default/file/data.csv')\
        .save()

#####Ignore

In [0]:
df.write.format('csv')\
    .mode('Ignore')\
    .option('path','dbfs:/Volumes/workspace/default/file/data.csv')\
        .save()

###Parquet

In [0]:
df.write.format('parquet')\
    .mode('Overwrite')\
    .option('path','dbfs:/Volumes/workspace/default/file/data.csv')\
        .save()

### table


In [0]:
df.write.format('delta')\
    .mode('Overwrite')\
        .saveAsTable('my_table')

In [0]:
spark.sql("DESCRIBE DETAIL my_table").show(truncate=False)

In [0]:
%sql
SELECT * FROM workspace.default.my_table;

In [0]:
df.display()

###Spark SQL

Using dataframe create temp views 


In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view

Convert back to data frame

In [0]:
df_sql = spark.sql("SELECT * FROM my_view")

In [0]:
df_sql.display()
